In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, accuracy_score, precision_score, recall_score, f1_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from scipy.stats import randint, uniform
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances
import time
import warnings
warnings.filterwarnings('ignore')

In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 15.0 MB/s eta 0:00:00


In [4]:
print("="*80)
print("LOADING LENDING CLUB LOAN DEFAULT DATASET")
print("="*80)

# Using a sample lending dataset from online source
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"

# Backup: German Credit Data
try:
    data = pd.read_excel(url, header=1)
    data.columns = ['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
                    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
                    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
                    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
                    'default']
    data = data.drop('ID', axis=1)
    print("Dataset: Credit Card Default Data")
except:
    # Fallback to German Credit Data
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
    column_names = ['account_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
                    'savings', 'employment', 'installment_rate', 'personal_status', 'debtors',
                    'residence', 'property', 'age', 'other_plans', 'housing', 'existing_credits',
                    'job', 'dependents', 'telephone', 'foreign_worker', 'class']
    data = pd.read_csv(url, sep=' ', names=column_names)
    data = data.rename(columns={'class': 'default'})
    data['default'] = data['default'].map({1: 0, 2: 1})
    print("Dataset: German Credit Data")

print(f"\nDataset Shape: {data.shape}")
print(f"Target Distribution:\n{data['default'].value_counts()}")
print(f"Target Ratio: {data['default'].value_counts(normalize=True)}")

# Prepare features and target
X = data.drop('default', axis=1)
y = data['default']

# Encode categorical variables if any
for col in X.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining Set Size: {X_train.shape}")
print(f"Test Set Size: {X_test.shape}")

LOADING LENDING CLUB LOAN DEFAULT DATASET
Dataset: Credit Card Default Data

Dataset Shape: (30000, 24)
Target Distribution:
default
0    23364
1     6636
Name: count, dtype: int64
Target Ratio: default
0    0.7788
1    0.2212
Name: proportion, dtype: float64

Training Set Size: (24000, 23)
Test Set Size: (6000, 23)


In [ ]:
# ============================================================================
# METHOD 1: GRID SEARCH CV
# ============================================================================
print("\n" + "="*80)
print("METHOD 1: GRID SEARCH CV (Exhaustive Search)")
print("="*80)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"\nParameter Grid:")
for param, values in param_grid.items():
    print(f"  {param:20s}: {values}")
print(f"\nTotal Combinations: {total_combinations}")

# Start Grid Search
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    verbose=2,
    n_jobs=-1,
    return_train_score=True
)

print("\n[GRID SEARCH] Starting training...")
start_time = time.time()
grid_search.fit(X_train, y_train)
grid_time = time.time() - start_time

print(f"\n[GRID SEARCH] Training completed in {grid_time:.2f} seconds")
print(f"\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param:20s}: {value}")

print(f"\nBest CV ROC-AUC Score: {grid_search.best_score_:.4f}")

# Evaluate on test set
grid_pred = grid_search.best_estimator_.predict(X_test)
grid_proba = grid_search.best_estimator_.predict_proba(X_test)[:, 1]
grid_test_score = roc_auc_score(y_test, grid_proba)

print(f"Test ROC-AUC Score: {grid_test_score:.4f}")

# Store results
grid_results = pd.DataFrame(grid_search.cv_results_)
print(f"\nTop 5 Parameter Combinations:")
print(grid_results[['params', 'mean_test_score', 'std_test_score']].nlargest(5, 'mean_test_score'))



METHOD 1: GRID SEARCH CV (Exhaustive Search)

Parameter Grid:
  n_estimators        : [100, 200, 300]
  max_depth           : [10, 20, 30, None]
  min_samples_split   : [2, 5, 10]
  min_samples_leaf    : [1, 2, 4]
  max_features        : ['sqrt', 'log2']
  bootstrap           : [True, False]

Total Combinations: 432

[GRID SEARCH] Starting training...
Fitting 5 folds for each of 432 candidates, totalling 2160 fits


In [ ]:
# ============================================================================
# METHOD 2: RANDOMIZED SEARCH CV
# ============================================================================
print("\n" + "="*80)
print("METHOD 2: RANDOMIZED SEARCH CV (Random Sampling)")
print("="*80)

param_distributions = {
    'n_estimators': randint(50, 500),
    'max_depth': [5, 10, 15, 20, 25, 30, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
    'bootstrap': [True, False],
    'min_impurity_decrease': uniform(0, 0.1),
    'max_leaf_nodes': [None, 10, 20, 50, 100],
    'class_weight': [None, 'balanced']
}

print(f"\nParameter Distributions:")
for param, dist in param_distributions.items():
    print(f"  {param:25s}: {dist}")

n_iter = 100
print(f"\nNumber of Iterations: {n_iter}")

# Start Randomized Search
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=5,
    scoring='roc_auc',
    verbose=2,
    n_jobs=-1,
    random_state=42,
    return_train_score=True
)

print("\n[RANDOM SEARCH] Starting training...")
start_time = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - start_time

print(f"\n[RANDOM SEARCH] Training completed in {random_time:.2f} seconds")
print(f"\nBest Parameters:")
for param, value in random_search.best_params_.items():
    print(f"  {param:20s}: {value}")

print(f"\nBest CV ROC-AUC Score: {random_search.best_score_:.4f}")

# Evaluate on test set
random_pred = random_search.best_estimator_.predict(X_test)
random_proba = random_search.best_estimator_.predict_proba(X_test)[:, 1]
random_test_score = roc_auc_score(y_test, random_proba)

print(f"Test ROC-AUC Score: {random_test_score:.4f}")

# Store results
random_results = pd.DataFrame(random_search.cv_results_)
print(f"\nTop 5 Parameter Combinations:")
print(random_results[['params', 'mean_test_score', 'std_test_score']].nlargest(5, 'mean_test_score'))

In [ ]:
# ============================================================================
# METHOD 3: OPTUNA (Bayesian Optimization)
# ============================================================================
print("\n" + "="*80)
print("METHOD 3: OPTUNA (Bayesian Optimization)")
print("="*80)

def objective(trial):
    """Optuna objective function"""

    # Suggest hyperparameters
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_categorical('max_depth', [5, 10, 15, 20, 25, 30, None]),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5, 0.7]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 0.1),
        'max_leaf_nodes': trial.suggest_categorical('max_leaf_nodes', [None, 10, 20, 50, 100]),
        'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
        'random_state': 42,
        'n_jobs': -1
    }

    # Create model
    model = RandomForestClassifier(**params)

    # Cross-validation score
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)

    return cv_scores.mean()

# Create study
print("\nCreating Optuna study with TPE sampler...")
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# Optimize
n_trials = 100
print(f"Number of Trials: {n_trials}")
print("\n[OPTUNA] Starting optimization...")

start_time = time.time()
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
optuna_time = time.time() - start_time

print(f"\n[OPTUNA] Optimization completed in {optuna_time:.2f} seconds")
print(f"\nBest Trial: {study.best_trial.number}")
print(f"Best CV ROC-AUC Score: {study.best_value:.4f}")
print(f"\nBest Parameters:")
for param, value in study.best_params.items():
    print(f"  {param:20s}: {value}")

# Train final model with best parameters
optuna_model = RandomForestClassifier(**study.best_params, random_state=42, n_jobs=-1)
optuna_model.fit(X_train, y_train)

# Evaluate on test set
optuna_pred = optuna_model.predict(X_test)
optuna_proba = optuna_model.predict_proba(X_test)[:, 1]
optuna_test_score = roc_auc_score(y_test, optuna_proba)

print(f"Test ROC-AUC Score: {optuna_test_score:.4f}")

In [ ]:
# ============================================================================
# COMPARISON OF ALL METHODS
# ============================================================================
print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON")
print("="*80)

methods = {
    'Grid Search CV': {
        'model': grid_search.best_estimator_,
        'time': grid_time,
        'cv_score': grid_search.best_score_,
        'test_score': grid_test_score,
        'n_combinations': total_combinations
    },
    'Random Search CV': {
        'model': random_search.best_estimator_,
        'time': random_time,
        'cv_score': random_search.best_score_,
        'test_score': random_test_score,
        'n_combinations': n_iter
    },
    'Optuna (Bayesian)': {
        'model': optuna_model,
        'time': optuna_time,
        'cv_score': study.best_value,
        'test_score': optuna_test_score,
        'n_combinations': n_trials
    }
}

# Create comparison dataframe
comparison_data = []
for method_name, method_data in methods.items():
    model = method_data['model']
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    comparison_data.append({
        'Method': method_name,
        'Time (s)': method_data['time'],
        'Combinations': method_data['n_combinations'],
        'CV ROC-AUC': method_data['cv_score'],
        'Test ROC-AUC': method_data['test_score'],
        'Test Accuracy': accuracy_score(y_test, y_pred),
        'Test Precision': precision_score(y_test, y_pred),
        'Test Recall': recall_score(y_test, y_pred),
        'Test F1': f1_score(y_test, y_pred)
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

# ============================================================================
# VISUALIZATIONS
# ============================================================================
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

# 1. Time vs Performance Comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Time comparison
axes[0, 0].bar(comparison_df['Method'], comparison_df['Time (s)'], color=['#3498db', '#e74c3c', '#2ecc71'])
axes[0, 0].set_ylabel('Time (seconds)', fontsize=11)
axes[0, 0].set_title('Training Time Comparison', fontsize=13, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=15)
for i, v in enumerate(comparison_df['Time (s)']):
    axes[0, 0].text(i, v + 0.5, f'{v:.1f}s', ha='center', fontweight='bold')

# ROC-AUC comparison
x = np.arange(len(comparison_df))
width = 0.35
axes[0, 1].bar(x - width/2, comparison_df['CV ROC-AUC'], width, label='CV Score', color='skyblue')
axes[0, 1].bar(x + width/2, comparison_df['Test ROC-AUC'], width, label='Test Score', color='lightcoral')
axes[0, 1].set_ylabel('ROC-AUC Score', fontsize=11)
axes[0, 1].set_title('Cross-Validation vs Test Performance', fontsize=13, fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(comparison_df['Method'], rotation=15)
axes[0, 1].legend()
axes[0, 1].set_ylim([0.5, 1.0])

# All metrics comparison
metrics = ['Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1']
x = np.arange(len(metrics))
width = 0.25
for i, method in enumerate(comparison_df['Method']):
    values = comparison_df.iloc[i][metrics].values
    axes[1, 0].bar(x + i*width, values, width, label=method)
axes[1, 0].set_ylabel('Score', fontsize=11)
axes[1, 0].set_title('All Performance Metrics Comparison', fontsize=13, fontweight='bold')
axes[1, 0].set_xticks(x + width)
axes[1, 0].set_xticklabels(metrics, rotation=15)
axes[1, 0].legend(fontsize=9)
axes[1, 0].set_ylim([0, 1.0])

# ROC Curves
for method_name, method_data in methods.items():
    y_proba = method_data['model'].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[1, 1].plot(fpr, tpr, linewidth=2.5, label=f'{method_name} (AUC={auc:.3f})')

axes[1, 1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[1, 1].set_xlabel('False Positive Rate', fontsize=11)
axes[1, 1].set_ylabel('True Positive Rate', fontsize=11)
axes[1, 1].set_title('ROC Curves Comparison', fontsize=13, fontweight='bold')
axes[1, 1].legend(loc='lower right', fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/comparison_overview.png', dpi=150, bbox_inches='tight')
print("✓ Saved: comparison_overview.png")

# 2. Optuna Optimization History
try:
    fig = optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.title('Optuna Optimization History', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/home/claude/optuna_history.png', dpi=150, bbox_inches='tight')
    print("✓ Saved: optuna_history.png")
    plt.close()
except:
    print("! Could not generate Optuna history plot")

# 3. Optuna Parameter Importances
try:
    fig = optuna.visualization.matplotlib.plot_param_importances(study)
    plt.title('Optuna Hyperparameter Importances', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/home/claude/optuna_param_importance.png', dpi=150, bbox_inches='tight')
    print("✓ Saved: optuna_param_importance.png")
    plt.close()
except:
    print("! Could not generate Optuna parameter importance plot")

# 4. Grid Search vs Random Search Convergence
plt.figure(figsize=(12, 6))

# Grid Search scores
grid_scores_sorted = grid_results.sort_values('mean_test_score', ascending=False)['mean_test_score'].values
plt.subplot(1, 2, 1)
plt.plot(range(1, len(grid_scores_sorted)+1), grid_scores_sorted, marker='o', markersize=3, linewidth=1.5)
plt.axhline(y=grid_search.best_score_, color='r', linestyle='--', label=f'Best: {grid_search.best_score_:.4f}')
plt.xlabel('Iteration (sorted by score)', fontsize=11)
plt.ylabel('CV ROC-AUC Score', fontsize=11)
plt.title('Grid Search Convergence', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

# Random Search scores
random_scores_sorted = random_results.sort_values('mean_test_score', ascending=False)['mean_test_score'].values
plt.subplot(1, 2, 2)
plt.plot(range(1, len(random_scores_sorted)+1), random_scores_sorted, marker='o', markersize=3, linewidth=1.5, color='orange')
plt.axhline(y=random_search.best_score_, color='r', linestyle='--', label=f'Best: {random_search.best_score_:.4f}')
plt.xlabel('Iteration (sorted by score)', fontsize=11)
plt.ylabel('CV ROC-AUC Score', fontsize=11)
plt.title('Random Search Convergence', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/convergence_comparison.png', dpi=150, bbox_inches='tight')
print("✓ Saved: convergence_comparison.png")

# ============================================================================
# DETAILED CLASSIFICATION REPORTS
# ============================================================================
print("\n" + "="*80)
print("DETAILED CLASSIFICATION REPORTS")
print("="*80)

for method_name, method_data in methods.items():
    print(f"\n{'='*80}")
    print(f"{method_name}")
    print(f"{'='*80}")
    model = method_data['model']
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

# ============================================================================
# KEY INSIGHTS
# ============================================================================
print("\n" + "="*80)
print("KEY INSIGHTS & RECOMMENDATIONS")
print("="*80)

best_method = comparison_df.loc[comparison_df['Test ROC-AUC'].idxmax(), 'Method']
fastest_method = comparison_df.loc[comparison_df['Time (s)'].idxmin(), 'Method']

print(f"""
1. PERFORMANCE:
   - Best performing method: {best_method}
   - Best Test ROC-AUC: {comparison_df['Test ROC-AUC'].max():.4f}

2. EFFICIENCY:
   - Fastest method: {fastest_method}
   - Training time: {comparison_df['Time (s)'].min():.2f} seconds

3. METHOD COMPARISON:
   - Grid Search: Exhaustive but slowest, tested {total_combinations} combinations
   - Random Search: Good balance, tested {n_iter} random combinations
   - Optuna: Smart sampling using Bayesian optimization, tested {n_trials} trials

4. RECOMMENDATIONS:
   - For small search spaces (<100 combinations): Use Grid Search
   - For medium search spaces (100-1000): Use Random Search
   - For large/complex spaces or limited time: Use Optuna
   - For production: Use Optuna for best performance/time tradeoff

5. HYPERPARAMETER INSIGHTS:
   Grid Search best params: {grid_search.best_params_}
   Random Search best params: {random_search.best_params_}
   Optuna best params: {study.best_params}
""")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print(f"\nGenerated files:")
print("  - comparison_overview.png")
print("  - optuna_history.png")
print("  - optuna_param_importance.png")
print("  - convergence_comparison.png")